# LangServe vs FastAPI: Serving LLM Applications

---

## What This Notebook Covers

This notebook demonstrates the difference between:

- **Raw FastAPI** — manually wiring HTTP endpoints for LLM chains
- **LangServe** — a purpose-built serving layer on top of FastAPI, designed specifically for LangChain runnables

We build the same chain twice: once with plain FastAPI, once with LangServe. The contrast reveals exactly what LangServe abstracts away and why it exists.

**Model used:** `gpt-4o-mini`  
**Runtime:** Google Colab compatible

## 1. Installation

In [ ]:
!pip install -q langchain langchain-openai langserve fastapi uvicorn pydantic requests nest-asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 16.1 MB/s eta 0:00:00


## 2. Environment Setup

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


## 3. Build the Core Chain (Shared)

We define one LangChain chain that both approaches will serve. It takes a topic and returns a concise technical explanation.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define the prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a precise technical educator. Explain concepts clearly and concisely in 3-4 sentences."),
    ("human", "Explain the concept: {topic}")
])

# Use gpt-4o-mini
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Compose the LCEL chain
chain = prompt | llm | StrOutputParser()

# Verify it works directly
response = chain.invoke({"topic": "vector databases"})
print("Direct chain output:")
print(response)

Direct chain output:
Vector databases are specialized data storage systems designed to handle and query high-dimensional vector data, which is often generated from machine learning models, particularly in natural language processing and computer vision. Unlike traditional databases that use structured data, vector databases store data as multi-dimensional vectors, allowing for efficient similarity searches and nearest neighbor queries. They utilize techniques such as indexing and approximate nearest neighbor (ANN) algorithms to quickly retrieve relevant vectors based on distance metrics. This makes them particularly useful for applications like recommendation systems, image retrieval, and semantic search.


## 4. Approach 1 — Raw FastAPI

### What you must do manually

When you use plain FastAPI to serve a LangChain chain, you are responsible for:

1. Defining request/response Pydantic schemas
2. Writing an `/invoke` endpoint
3. Writing a `/stream` endpoint with SSE handling
4. Writing a `/batch` endpoint
5. Error handling and input validation
6. No auto-generated playground or schema documentation beyond OpenAPI basics

The code below shows the full manual implementation.

In [ ]:
import nest_asyncio
import asyncio
import threading
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import List

nest_asyncio.apply()

# --- Manual Schema Definition ---
class InvokeRequest(BaseModel):
    topic: str

class InvokeResponse(BaseModel):
    output: str

class BatchRequest(BaseModel):
    topics: List[str]

class BatchResponse(BaseModel):
    outputs: List[str]

# --- FastAPI App ---
fastapi_app = FastAPI(
    title="Manual FastAPI LLM Server",
    description="Serving a LangChain chain manually — all boilerplate written by hand"
)

# --- Endpoint 1: /invoke ---
# You must write this endpoint yourself
@fastapi_app.post("/explain/invoke", response_model=InvokeResponse)
async def invoke_endpoint(request: InvokeRequest):
    try:
        result = await chain.ainvoke({"topic": request.topic})
        return InvokeResponse(output=result)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# --- Endpoint 2: /stream ---
# You must implement SSE streaming manually
@fastapi_app.post("/explain/stream")
async def stream_endpoint(request: InvokeRequest):
    async def token_generator():
        try:
            async for chunk in chain.astream({"topic": request.topic}):
                yield f"data: {chunk}\n\n"
        except Exception as e:
            yield f"data: ERROR: {str(e)}\n\n"
    return StreamingResponse(token_generator(), media_type="text/event-stream")

# --- Endpoint 3: /batch ---
# You must write batch logic manually
@fastapi_app.post("/explain/batch", response_model=BatchResponse)
async def batch_endpoint(request: BatchRequest):
    try:
        inputs = [{"topic": t} for t in request.topics]
        results = await chain.abatch(inputs)
        return BatchResponse(outputs=results)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("FastAPI app defined with 3 manually written endpoints.")
print("Endpoints: /explain/invoke, /explain/stream, /explain/batch")

FastAPI app defined with 3 manually written endpoints.
Endpoints: /explain/invoke, /explain/stream, /explain/batch


In [ ]:
import requests
import time

# Run FastAPI server in background thread
def run_fastapi():
    uvicorn.run(fastapi_app, host="0.0.0.0", port=8001, log_level="error")

fastapi_thread = threading.Thread(target=run_fastapi, daemon=True)
fastapi_thread.start()
time.sleep(3)  # Wait for server to start

# Test the invoke endpoint
response = requests.post(
    "http://localhost:8001/explain/invoke",
    json={"topic": "attention mechanism in transformers"}
)

print("FastAPI /invoke response:")
print(response.json())

FastAPI /invoke response:
{'output': 'The attention mechanism in transformers allows the model to weigh the importance of different words in a sequence when generating an output. It computes a set of attention scores that determine how much focus to place on each word relative to others, enabling the model to capture contextual relationships effectively. This is achieved through three components: queries, keys, and values, which are derived from the input embeddings. By using these components, the model can dynamically adjust its focus, improving its ability to understand and generate language.'}


In [ ]:
# Test batch endpoint
batch_response = requests.post(
    "http://localhost:8001/explain/batch",
    json={"topics": ["RLHF", "RAG"]}
)

print("FastAPI /batch response:")
for i, output in enumerate(batch_response.json()["outputs"]):
    print(f"\n[{i+1}] {output}")

FastAPI /batch response:

[1] RLHF stands for Reinforcement Learning from Human Feedback. It is a machine learning approach that combines reinforcement learning (RL) with feedback provided by humans to improve the performance of AI models. In this process, a model is initially trained using traditional supervised learning, and then it is fine-tuned using reinforcement learning, where human evaluators provide feedback on the model's outputs. This feedback helps the model learn to generate responses that align more closely with human preferences and values, enhancing its overall effectiveness in tasks like natural language processing.

[2] RAG stands for Retrieval-Augmented Generation, a technique that enhances the capabilities of language models by integrating external information retrieval. In this approach, a model first retrieves relevant documents or data from a knowledge base based on a user's query. It then uses this retrieved information to generate more accurate and contextually

## 5. Approach 2 — LangServe

### What LangServe does for you

LangServe wraps any LangChain Runnable and automatically generates:

| Feature | FastAPI (manual) | LangServe |
|---|---|---|
| `/invoke` endpoint | Write yourself | Auto-generated |
| `/stream` endpoint | Write yourself | Auto-generated |
| `/batch` endpoint | Write yourself | Auto-generated |
| `/stream_log` endpoint | Write yourself | Auto-generated |
| Input/output schema | Write yourself | Inferred from chain |
| Interactive playground | Not included | `/playground` route |
| Type validation | Pydantic manually | Automatic |

**The key function:** `add_routes(app, chain, path="/explain")`  
One line replaces all three manual endpoint definitions.

In [ ]:
from fastapi import FastAPI
from langserve import add_routes

# LangServe app — notice how minimal this is
langserve_app = FastAPI(
    title="LangServe LLM Server",
    description="The same chain, served via LangServe"
)

# One line replaces all three manual endpoints
add_routes(
    langserve_app,
    chain,
    path="/explain"
)

print("LangServe app defined.")
print("Auto-generated endpoints:")
print("  POST /explain/invoke")
print("  POST /explain/stream")
print("  POST /explain/batch")
print("  POST /explain/stream_log")
print("  GET  /explain/playground")
print("  GET  /explain/input_schema")
print("  GET  /explain/output_schema")

LangServe app defined.
Auto-generated endpoints:
  POST /explain/invoke
  POST /explain/stream
  POST /explain/batch
  POST /explain/stream_log
  GET  /explain/playground
  GET  /explain/input_schema
  GET  /explain/output_schema


In [ ]:
# Run LangServe server
def run_langserve():
    uvicorn.run(langserve_app, host="0.0.0.0", port=8002, log_level="error")

langserve_thread = threading.Thread(target=run_langserve, daemon=True)
langserve_thread.start()
time.sleep(3)

# LangServe expects the input wrapped in {"input": ...}
response = requests.post(
    "http://localhost:8002/explain/invoke",
    json={"input": {"topic": "attention mechanism in transformers"}}
)

print("LangServe /invoke response:")
print(response.json())


     __          ___      .__   __.   _______      _______. _______ .______     ____    ____  _______
    |  |        /   \     |  \ |  |  /  _____|    /       ||   ____||   _  \    \   \  /   / |   ____|
    |  |       /  ^  \    |   \|  | |  |  __     |   (----`|  |__   |  |_)  |    \   \/   /  |  |__
    |  |      /  /_\  \   |  . `  | |  | |_ |     \   \    |   __|  |      /      \      /   |   __|
    |  `----./  _____  \  |  |\   | |  |__| | .----)   |   |  |____ |  |\  \----.  \    /    |  |____
    |_______/__/     \__\ |__| \__|  \______| |_______/    |_______|| _| `._____|   \__/     |_______|
    
LANGSERVE: Playground for chain "/explain/" is live at:
LANGSERVE:  │
LANGSERVE:  └──> /explain/playground/
LANGSERVE:
LANGSERVE: See all available routes at /docs/
LangServe /invoke response:
{'output': 'The attention mechanism in transformers allows the model to weigh the importance of different words in a sequence when processing input data. It computes a set of attention score

In [ ]:
# LangServe auto-generates input/output schemas — no Pydantic models needed from us
input_schema = requests.get("http://localhost:8002/explain/input_schema")
output_schema = requests.get("http://localhost:8002/explain/output_schema")

import json
print("Auto-inferred Input Schema:")
print(json.dumps(input_schema.json(), indent=2))

print("\nAuto-inferred Output Schema:")
print(json.dumps(output_schema.json(), indent=2))

Auto-inferred Input Schema:
{
  "properties": {
    "topic": {
      "title": "Topic",
      "type": "string"
    }
  },
  "required": [
    "topic"
  ],
  "title": "PromptInput",
  "type": "object"
}

Auto-inferred Output Schema:
{
  "title": "StrOutputParserOutput",
  "type": "string"
}


In [ ]:
# LangServe batch — same structure across all endpoints
batch_response = requests.post(
    "http://localhost:8002/explain/batch",
    json={"inputs": [{"topic": "RLHF"}, {"topic": "RAG"}]}
)

print("LangServe /batch response:")
for i, output in enumerate(batch_response.json()["output"]):
    print(f"\n[{i+1}] {output}")

LangServe /batch response:

[1] RLHF stands for Reinforcement Learning from Human Feedback. It is a machine learning approach that combines reinforcement learning (RL) with feedback provided by humans to improve the performance of AI models. In this process, a model is initially trained using traditional supervised learning, and then it is fine-tuned using reinforcement learning, where human evaluators provide feedback on the model's outputs. This feedback helps the model learn to generate more desirable responses, aligning its behavior with human preferences.

[2] RAG stands for Retrieval-Augmented Generation, a technique that enhances the capabilities of language models by integrating external information retrieval. In this approach, a model first retrieves relevant documents or data from a knowledge base based on a query, and then uses this information to generate more accurate and contextually relevant responses. This method improves the model's performance on tasks requiring up-to

## 6. Using the LangServe RemoteRunnable Client

LangServe also ships a client — `RemoteRunnable` — that lets you call a remote LangServe server as if it were a local chain. This is the correct way to consume a LangServe endpoint in client code.

In [ ]:
from langserve import RemoteRunnable

# Treat the remote server exactly like a local chain
remote_chain = RemoteRunnable("http://localhost:8002/explain")

# invoke
result = remote_chain.invoke({"topic": "gradient descent"})
print("RemoteRunnable invoke:")
print(result)

# stream — identical API to a local chain
print("\nRemoteRunnable stream:")
for chunk in remote_chain.stream({"topic": "tokenization"}):
    print(chunk, end="", flush=True)
print()

RemoteRunnable invoke:
Gradient descent is an optimization algorithm used to minimize a function by iteratively moving towards the steepest descent, or the negative gradient, of the function. In the context of machine learning, it is commonly used to minimize the loss function, which measures the difference between predicted and actual outcomes. The algorithm updates the model parameters by taking small steps proportional to the negative of the gradient, controlled by a learning rate. This process continues until the algorithm converges to a minimum point, ideally the global minimum, or until a specified number of iterations is reached.

RemoteRunnable stream:
Tokenization is the process of converting sensitive data, such as credit card numbers or personal information, into unique identification symbols called tokens. These tokens retain essential information about the data without compromising its security, as they cannot be reverse-engineered to reveal the original data. Tokenization

## 7. Direct Comparison Summary

The table below captures every dimension of difference between the two approaches.

In [ ]:
import pandas as pd

comparison = {
    "Dimension": [
        "Lines of code to serve invoke+stream+batch",
        "Input/output schema",
        "Streaming (SSE)",
        "stream_log endpoint",
        "Interactive playground",
        "Client library",
        "LangChain integration",
        "Custom endpoint logic",
        "Production flexibility",
        "Learning curve"
    ],
    "Raw FastAPI": [
        "~50 lines",
        "Write Pydantic models manually",
        "Write SSE generator manually",
        "Write manually",
        "Not included",
        "Use requests/httpx",
        "Manual — call chain inside handler",
        "Full control",
        "Very high — build anything",
        "Standard FastAPI"
    ],
    "LangServe": [
        "1 line (add_routes)",
        "Auto-inferred from chain",
        "Auto-generated",
        "Auto-generated",
        "/playground route included",
        "RemoteRunnable (chain-like API)",
        "First-class — designed for runnables",
        "Less — opinionated structure",
        "Good for standard LLM serving",
        "Minimal if you know LangChain"
    ]
}

df = pd.DataFrame(comparison)
df.set_index("Dimension", inplace=True)
print(df.to_string())

                                                                   Raw FastAPI                             LangServe
Dimension                                                                                                           
Lines of code to serve invoke+stream+batch                           ~50 lines                   1 line (add_routes)
Input/output schema                             Write Pydantic models manually              Auto-inferred from chain
Streaming (SSE)                                   Write SSE generator manually                        Auto-generated
stream_log endpoint                                             Write manually                        Auto-generated
Interactive playground                                            Not included            /playground route included
Client library                                              Use requests/httpx       RemoteRunnable (chain-like API)
LangChain integration                       Manual — call chain 

## 8. When to Use Which

**Use LangServe when:**
- You are serving a LangChain chain or runnable
- You want all standard endpoints (invoke, stream, batch, stream_log) without boilerplate
- You want the interactive playground for testing during development
- You want clients to use `RemoteRunnable` for a consistent API

**Use raw FastAPI when:**
- Your serving logic is not a LangChain chain (custom models, non-LCEL pipelines)
- You need endpoints that do not map cleanly to invoke/stream/batch
- You need full control over request/response shapes, middleware, or auth
- You are mixing LLM endpoints with other API logic in one service

**Key insight:** LangServe is built on top of FastAPI. It does not replace it — it adds a thin, purpose-built layer for LangChain runnables. You can mix `add_routes` with custom FastAPI endpoints in the same app.

In [ ]:
# Practical pattern: mix LangServe routes with custom FastAPI endpoints
from fastapi import FastAPI
from langserve import add_routes

app = FastAPI(title="Mixed App")

# LangServe handles the LLM chain
add_routes(app, chain, path="/explain")

# Custom FastAPI endpoint alongside LangServe routes
@app.get("/health")
async def health_check():
    return {"status": "ok", "model": "gpt-4o-mini"}

@app.get("/models")
async def list_models():
    return {"available": ["gpt-4o-mini"]}

print("Mixed app routes:")
for route in app.routes:
    if hasattr(route, 'path'):
        print(f"  {route.path}")

Mixed app routes:
  /openapi.json
  /docs
  /docs/oauth2-redirect
  /redoc
  /explain/invoke
  /explain/c/{config_hash}/invoke
  /explain/batch
  /explain/c/{config_hash}/batch
  /explain/stream
  /explain/c/{config_hash}/stream
  /explain/stream_log
  /explain/c/{config_hash}/stream_log
  /explain/stream_events
  /explain/c/{config_hash}/stream_events
  /explain/input_schema
  /explain/c/{config_hash}/input_schema
  /explain/output_schema
  /explain/c/{config_hash}/output_schema
  /explain/config_schema
  /explain/c/{config_hash}/config_schema
  /explain/playground/{file_path:path}
  /explain/c/{config_hash}/playground/{file_path:path}
  /explain/token_feedback
  /explain/invoke
  /explain/c/{config_hash}/invoke
  /explain/batch
  /explain/c/{config_hash}/batch
  /explain/stream
  /explain/c/{config_hash}/stream
  /explain/stream_log
  /explain/c/{config_hash}/stream_log
  /explain/stream_events
  /explain/c/{config_hash}/stream_events
  /health
  /models


## Summary

LangServe is not a replacement for FastAPI — it is a productivity layer on top of it, purpose-built for LangChain runnables. The code reduction is real (50 lines to 1 line for standard serving), but the more important benefit is the consistency: every LangServe endpoint has the same request/response contract, the same streaming behavior, and the same schema introspection, which makes client code simpler and more reliable.

Raw FastAPI remains the right choice when your use case does not fit the invoke/stream/batch pattern or when you need full endpoint control.